In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [ ]:
transform = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor()
])

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for c in classes:
            class_path = os.path.join(root_dir, c)

            for fname in os.listdir(class_path):
                self.samples.append(
                    (os.path.join(class_path, fname),
                     self.class_to_idx[c])
                )

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
train_dataset = FaceDataset("/kaggle/input/dataset-face-1/train",
                            transform=transform)
val_dataset = ImageFolder("/kaggle/input/dataset-face-1/val", transform)
test_dataset = ImageFolder("/kaggle/input/dataset-face-1/test", transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
class SiameseNet(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()

        base = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.DEFAULT
        )

        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)

        in_feat = base.classifier[0].in_features
        self.fc = nn.Linear(in_feat, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        x = F.normalize(x, dim=1)
        return x

In [ ]:
class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.5):
        super().__init__()

        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim)
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings)
        weight = F.normalize(self.weight)

        cosine = F.linear(embeddings, weight)

        theta = torch.acos(torch.clamp(cosine, -1+1e-7, 1-1e-7))
        target_logits = torch.cos(theta + self.m)

        one_hot = F.one_hot(labels, num_classes=cosine.size(1)).float()

        logits = cosine * (1 - one_hot) + target_logits * one_hot
        logits *= self.s

        loss = F.cross_entropy(logits, labels)
        return loss

In [ ]:
class SiameseNet(nn.Module):
    def __init__(self, embedding_dim=256):
        super(SiameseNet, self).__init__()

        mobilenet = models.mobilenet_v3_large(weights="DEFAULT")                    

        self.feature_extractor = mobilenet.features

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        last_channel = mobilenet.classifier[0].in_features
        self.fc = nn.Linear(last_channel, embedding_dim)

        self.dropout = nn.Dropout(0.5)

    def forward_once(self, x):
        x = self.feature_extractor(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc(x)))
        x = F.normalize(x, p=2, dim=1)
        return x

    def forward(self, anchor, positive, negative):
        out_anchor = self.forward_once(anchor)
        out_positive = self.forward_once(positive)
        out_negative = self.forward_once(negative)
        return out_anchor, out_positive, out_negative


In [ ]:
class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2)
        neg_dist = F.pairwise_distance(anchor, negative, p=2)

        losses = F.relu(pos_dist - neg_dist + self.margin)
        return losses.mean()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def recall_at_k(embeddings, labels, k=1):
    embeddings = F.normalize(embeddings)

    sim = embeddings @ embeddings.T

    _, indices = sim.topk(k + 1, dim=1)

    indices = indices[:, 1:]
    retrieved = labels[indices]

    correct = (retrieved == labels.unsqueeze(1)).any(dim=1)
    return correct.float().mean().item()

In [ ]:
def train_arcface(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=20,
):

    model.to(device)

    for epoch in range(epochs):

        # ===== TRAIN =====
        model.train()
        running_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)

            emb = model(imgs)
            loss = criterion(emb, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader)

        # ===== VALIDATION =====
        model.eval()
        all_emb, all_labels = [], []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                emb = model(imgs)

                all_emb.append(emb.cpu())
                all_labels.append(labels)

        all_emb = torch.cat(all_emb)
        all_labels = torch.cat(all_labels)

        r1 = recall_at_k(all_emb, all_labels, 1)
        r5 = recall_at_k(all_emb, all_labels, 5)

        print(f"Loss: {train_loss:.4f} | R@1: {r1:.4f} | R@5: {r5:.4f}")

In [ ]:
num_classes = len(train_dataset.class_to_idx)

model = SiameseNet(embedding_dim=256).to(device)

criterion = ArcFaceLoss(
    embedding_dim=256,
    num_classes=num_classes,
    s=30,
    m=0.5
).to(device)

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=1e-4
)

# train_arcface(
#     model,
#     train_loader,
#     val_loader,
#     criterion,
#     optimizer,
#     device,
#     epochs=20
# )

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/siamese_model_tripletloss_1.pth")
model.load_state_dict(torch.load("/kaggle/working/siamese_model_tripletloss_1.pth"))
model.eval()
print('ok')

In [ ]:
def predic_cosin_2_img(img_input1, img_input2):
    
    face1 = Image.open(img_input1).convert("RGB")
    face2 = Image.open(img_input2).convert("RGB")

    img1 = transform(face1).unsqueeze(0).to(device)
    img2 = transform(face2).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        embed_anchor, embed_target = model.forward(img1), model.forward(img2)

    # Cosine similarity
    cosin = torch.nn.functional.cosine_similarity(embed_anchor, embed_target).item()
    return cosin

In [ ]:
img1 = r"/kaggle/input/dataset-face-1/test/0000186/0000186_001.jpg"
img2 = r"/kaggle/input/dataset-face-1/test/0000186/0000186_029.jpg"

predic_cosin_2_img(img1, img2)

In [ ]:
import torch
import numpy as np

def extract_embeddings(model, dataloader, device):
    model.eval()

    embeddings = []
    labels = []

    with torch.no_grad():
        for imgs, lbls in dataloader:
            imgs = imgs.to(device)

            emb = model(imgs)

            embeddings.append(emb.cpu().numpy())
            labels.append(lbls.numpy())

    embeddings = np.concatenate(embeddings)
    labels = np.concatenate(labels)

    return embeddings, labels

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_pairs(embeddings, labels):

    sims = []
    targets = []

    N = len(embeddings)

    for i in range(N):
        for j in range(i+1, N):

            sim = cosine_similarity(
                embeddings[i].reshape(1,-1),
                embeddings[j].reshape(1,-1)
            )[0][0]

            sims.append(sim)

            if labels[i] == labels[j]:
                targets.append(1)
            else:
                targets.append(0)

    return np.array(sims), np.array(targets)

In [ ]:
from sklearn.metrics import roc_curve, auc

def evaluate_verification(similarities, targets):

    fpr, tpr, thresholds = roc_curve(targets, similarities)

    roc_auc = auc(fpr, tpr)

    fnr = 1 - tpr

    eer_threshold = thresholds[np.nanargmin(np.abs(fnr - fpr))]
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

    preds = similarities > eer_threshold
    acc = (preds == targets).mean()

    print("AUC:", roc_auc)
    print("EER:", eer)
    print("Best Threshold:", eer_threshold)
    print("Verification Accuracy:", acc)

    return fpr, tpr, roc_auc, eer, eer_threshold

In [ ]:
def test_siamese(model, dataloader, device):

    embeddings, labels = extract_embeddings(model, dataloader, device)

    sims, targets = compute_pairs(embeddings, labels)

    fpr, tpr, roc_auc, eer, threshold = evaluate_verification(sims, targets)

    return {
        "AUC": roc_auc,
        "EER": eer,
        "threshold": threshold
    }